# Two Layers That Beat The 2945 Farm

**TL;DR**: Stack two public micro-optimizations on Tschinkel's base. 346 wins, 52 losses across a 10-agent gauntlet.
Fork this, submit, and read on to learn how money-ledger diagnostics found what parameter tweaking couldn't.

---

**What this notebook does:**
1. Embeds the complete agent (no external datasets, no GPU, no internet)
2. Writes `main.py` and packages `submission.tar.gz`
3. Fork → Submit → Done

**Gauntlet Results (6 key opponents, 50 games each):**

| Opponent | W-L-T | Elo Rate |
|----------|-------|----------|
| **Tschinkel base** | 49-1 | **98%** |
| **lynnsakurai v2** | 47-3 | **94%** |
| **ahmed v48** | 47-3 | **94%** |
| **dmitriigluzdov** | 33-1-16T | **82%** |
| **tetsutani** | 31-19 | **62%** |
| **ahmed v49** | 29-21 | **58%** |

---

## How We Found It: Diagnose, Don't Guess

We tried **5 parameter tweaks** on Tschinkel's agent first. Every single one regressed:

| Tweak | Result vs Tschinkel | Verdict |
|-------|-------------------|---------|
| Opening BUY 25 (was 20) | **0W-50L** (every game -$1,730) | Cash destruction |
| RACEGATE wheat threshold 15 (was 25) | 2W-3L-45T (neutral) | No effect |
| SL3 hire cap | 3W-1L-46T (neutral) | Already optimized |
| RACE sell layer | 16% vs Tschinkel | Broke debt tracking |
| Combined opening+RACEGATE | 27W-23L (54%) | Worse than baseline |

**Lesson**: Tschinkel's 5,760 lines are tightly calibrated. Changing parameters cascades into regressions.

So we changed approach: **find agents that BEAT Tschinkel, diagnose WHY with a per-day money ledger, then extract their thin layers.**

---

## Discovery 1: The $50 Opening Exploit (dmitriigluzdov)

We ran Tschinkel against dmitriigluzdov's "A Smaller Market Shock" — **result: 1W-49L**.

Almost every loss was by **exactly $50**. The money ledger showed the entire gap appears at day 0 and persists unchanged for 30 days. No other action differences.

**The mechanism**: Tschinkel's opening does a wheat round trip (`BUY 20, SELL 15`) that loses ~$50 against certain opponent openings. dmitriigluzdov replaces it with `BUY 5 + BUY_SEED 1` and redirects idle workers to plant/harvest a temporary wheat field — turning wasted `PASS` actions into productive output.

**Our fix (HybridOpening)**: Keep Tschinkel's profitable round trip (it wins money against most opponents), but ADD the idle worker redirection. Best of both worlds. Cost: 1 wheat seed ($10).

---

## Discovery 2: The Queue Closure (lynnsakurai)

Tschinkel vs lynnsakurai's "Farming Score v2" — **result: 3W-27L-20T**.

The 20 ties are exact ($0 gap through all 30 days). In the 30 contested games, lynnsakurai wins 27. The money ledger showed:

- **Days 0-17: identical** ($0 gap — same tape, same execution)
- **Days 18-29: lynnsakurai pulls ahead** by ~$220

lynnsakurai doesn't add new sells. It **reorders existing sell orders within the market array**:

1. **Zeros out dead orders** — sell orders where there's nothing in the shed to sell
2. **Pulls executable sells into earlier slots** — filling the holes left by zeroed orders

Earlier slot position = executes first in the engine = gets the higher price before market saturation.

~$220 average gain from pure execution ordering. No new logic, no new sells — just better queue management.

---

## Validation: 300 Games Across 6 Top Opponents

We stacked both layers on Tschinkel's base and ran 50 games against each top opponent:

| Opponent | This Agent | Tschinkel Base | Improvement |
|----------|-----------|---------------|-------------|
| dmitriigluzdov (market shock) | **82%** | 2% | **+80** |
| lynnsakurai (farming v2) | **94%** | 26% | **+68** |
| Tschinkel base | **98%** | — | — |
| ahmed v48 | **94%** | 86% | +8 |
| tetsutani | 62% | 68% | -6 |
| ahmed v49 | 58% | 60% | -2 |

The tetsutani/v49 regression comes from the EQ closure reordering sells in a way these specific opponents exploit. On LB where dmitriigluzdov/lynnsakurai forks are common, the net effect is strongly positive.

---

## The Journey: 15 Generations

| Gen | Key Change | Best Matchup | Method |
|-----|-----------|-------------|--------|
| Pipe-1 | Base agent (v40) | 81.8% | Pipeline discovery |
| Pipe-4 | +C9 opening | LB 2730 | Pipeline |
| Pipe-7 | Wheat microstructure | 97.4% local | Pipeline + colosseum |
| Pipe-11 | v48 layer absorption | LB 2579 | Autoloop v4 |
| Pipe-12 | 4 reactive layers | 10% vs Tschinkel | All layers regressed |
| Pipe-13 | Tschinkel + EarlyCycle + EQ | 79% gauntlet | Layer stacking |
| **Pipe-15** | **Hybrid opening + EQ closure** | **85% gauntlet** | **Diagnose → Extract → Stack** |

The winning method: don't tweak parameters on an optimized agent. **Find agents that beat it, diagnose the mechanism, extract their thin layers, and stack them.**

---

## Build the Agent

No datasets, no internet, no GPU. The complete agent is embedded below.

In [ ]:
from pathlib import Path
import os, sys, time

WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

In [ ]:
import base64, gzip, hashlib

EXPECTED_SHA256 = 'ce9ec069d8c72e93751521e6344bbcd2421700f66388633a7b718d479501bc09'

# pipe-15: Tschinkel v9/4 base + HybridOpening (worker utilization) + EQ sell closure
# Compressed and base64-encoded. 6,001 lines, 866,969 bytes uncompressed.
AGENT_B64 = (
    "..."
)

source_bytes = gzip.decompress(base64.b64decode(AGENT_B64))
digest = hashlib.sha256(source_bytes).hexdigest()
assert digest == EXPECTED_SHA256, f'Hash mismatch: {digest}'

main_py = WORKDIR / 'main.py'
main_py.write_bytes(source_bytes)
print(f'Wrote {main_py} ({len(source_bytes):,} bytes, sha256 {digest[:12]}...)')

In [ ]:
import ast

# Verify the agent parses and has the expected entry point
tree = ast.parse(source_bytes.decode('utf-8', errors='replace'))
functions = [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
assert 'agent' in functions, f'No agent() function found! Functions: {functions[:10]}'
print(f'Verified: agent() found among {len(functions)} functions')

## Build the Submission Archive

In [ ]:
import gzip as gzip_mod, io, tarfile
from IPython.display import FileLink, display

ARCHIVE = WORKDIR / 'submission.tar.gz'

buffer = io.BytesIO()
with tarfile.open(fileobj=buffer, mode='w', format=tarfile.GNU_FORMAT) as tar:
    info = tarfile.TarInfo('main.py')
    info.size, info.mtime, info.mode = len(source_bytes), 0, 0o644
    tar.addfile(info, io.BytesIO(source_bytes))
ARCHIVE.write_bytes(gzip_mod.compress(buffer.getvalue(), mtime=0))

sz = ARCHIVE.stat().st_size
print(f'Created {ARCHIVE} ({sz:,} bytes / {sz/1024:.0f} KB)')

# Verify
with tarfile.open(ARCHIVE) as tar:
    names = tar.getnames()
    extracted = tar.extractfile('main.py').read()
    assert extracted == source_bytes, 'Content mismatch!'
    print(f'Contents: {names} — verified identical')

print('Ready to submit!')
display(FileLink(str(ARCHIVE)))

---

# Attribution and License

This agent is a derivative work. Full credit to the authors:

### Base agent: Thomas Tschinkel
- **The 2945 Farm** (v9/4): Route replayer chassis, RACEPX/RACE reservation, COURIER, CARROT, HERD layers
- Apache-2.0

### Layer 1: Dmitrii Gluzdov
- **A Smaller Market Shock**: Bounded alternative productive openings (EarlyCycle)
- Redirects idle workers to plant/harvest temporary wheat crop
- Apache-2.0

### Layer 2: lynnsakurai
- **Farming Score v2 — A Better Approach**: Effective Queue (EQ) sell closure
- Reorders sell orders within market array for better execution pricing
- Apache-2.0

### Upstream lineage (preserved in agent source):
- **Ahmed Berat Ozer**: V25–V49 progression, the public V39 tape that Tschinkel built on
- **yhay81**: Shop router action tapes
- **prvsiyan**: V221B/V224C production/timing lineage
- **aurax7**: Reactive reservation activation
- **tetsutani**: Demand-preserving turn sale timing
- **Rayk Kretzschmar, degnonguidi, leoprovorov, destbreso, lucifer19**: Additional layers and tooling

All upstream Apache-2.0 notices are retained in the agent source.

---
*15 generations. 5 failed parameter tweaks. 2 thin layers that actually worked. If you found this useful, an upvote helps!*